Rows are grouped and split by the target output image. The printed labels
"Unique subjects (targets)" and "N subjects" below are historical: they count unique target
output images, not verified semantic subject identities.

In [ ]:
from huggingface_hub import hf_hub_download
import json, re, random
from collections import Counter, defaultdict

random.seed(42)

path = hf_hub_download(repo_id="yzwang/X2I-subject-driven",
                       filename="grit-entity-new.jsonl", repo_type="dataset")

keywords = [
    "desktop computer","computer setup","laptop","tablet","smartphone",
    "mobile phone","cell phone","headphones","earphones","earbuds",
    "keyboard","computer mouse","game controller","gaming controller",
    "gaming console","vr headset","virtual reality headset",
]
# Substring match on the instruction. Residual false positives caught by this cue on the
# finished subset (a lower bound: musical keyboards whose caption omits the cue are missed),
# cue = the substring "music" (also matching musical / musician), no other device keyword:
# 408 of 40,479 rows (1.0%) are musical keyboards; 7 of the 488 test rows, all from
# triple/target3/11_3686.png. Left in place — every comparison is paired on uid, so those
# rows appear on both sides of every paired comparison; excluding them changes no verdict.
pat = re.compile("|".join(re.escape(k) for k in keywords), re.IGNORECASE)
KEEP_DIRS = {"single", "double", "triple"}

# Group rows by target scene = the output_image (all its 1/2/3-reference versions together)
groups   = defaultdict(list)
refcount = Counter()
for line in open(path, encoding="utf-8"):
    row = json.loads(line)
    if not pat.search(str(row.get("instruction", ""))):
        continue
    imgs, out = row.get("input_images", []), row.get("output_image", "")
    dirs = {p.split("/")[0] for p in imgs + [out] if p}
    if not dirs or not dirs.issubset(KEEP_DIRS):   # keep single/double/triple only
        continue
    groups[out].append(row)
    refcount[len(imgs)] += 1

targets = list(groups.keys())
random.shuffle(targets)
n_rows = sum(len(v) for v in groups.values())
print(f"Domain rows: {n_rows}")
print(f"Unique subjects (targets): {len(targets)}")
print(f"Reference-photo spread (rows with 1 / 2 / 3 photos): {dict(sorted(refcount.items()))}")

# Split by target scene so the same output image never appears in two splits
test_t, val_t, train_t = set(targets[:100]), set(targets[100:120]), set(targets[120:])

def dump(name, tset):
    rows = [r for t in tset for r in groups[t]]
    with open(f"{name}.jsonl", "w", encoding="utf-8") as f:
        for r in rows: f.write(json.dumps(r) + "\n")
    return len(rows)

print("\nSplit (no leakage):")
print(f"  train: {len(train_t)} subjects, {dump('train', train_t)} rows")
print(f"  val:   {len(val_t)} subjects, {dump('val', val_t)} rows")
print(f"  test:  {len(test_t)} subjects, {dump('test', test_t)} rows")

grit-entity-new.jsonl:   0%|          | 0.00/653M [00:00<?, ?B/s]

Domain rows: 40479
Unique subjects (targets): 8179
Reference-photo spread (rows with 1 / 2 / 3 photos): {1: 20122, 2: 16150, 3: 4207}

Split (no leakage):
  train: 8059 subjects, 39893 rows
  val:   20 subjects, 98 rows
  test:  100 subjects, 488 rows


In [ ]:
import os, subprocess
from huggingface_hub import HfApi
from google.colab import userdata

os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
repo = "yzwang/X2I-subject-driven"

files = HfApi().list_repo_files(repo, repo_type="dataset")
for d in ["single", "double", "triple"]:
    parts = sorted(f for f in files if f.startswith(f"{d}/") and ".tar.gz." in f)
    print(f"{d}: {len(parts)} parts  (first: {parts[0]})")

# Peek at how files are named INSIDE the single/ archive (streams only a few MB)
first = sorted(f for f in files if f.startswith("single/") and ".tar.gz." in f)[0]
url = f"https://huggingface.co/datasets/{repo}/resolve/main/{first}"
cmd = f'curl -sL -H "Authorization: Bearer $HF_TOKEN" "{url}" | tar -tz 2>/dev/null | head -20'
print(f"\nSample file names inside {first}:")
print(subprocess.run(cmd, shell=True, capture_output=True, text=True, env=os.environ).stdout)

single: 109 parts  (first: single/single.tar.gz.aa)
double: 90 parts  (first: double/double.tar.gz.aa)
triple: 139 parts  (first: triple/triple.tar.gz.aa)

Sample file names inside single/single.tar.gz.aa:
input/
input/7/
input/7/383-mat-2-0.png
input/7/2123-mat-2-2.png
input/7/591-mat-2-1.png
input/7/3294-mat-1-0_up.png
input/7/342-mat-2-1.png
input/7/1559-mat-1-0_up.png
input/7/1164-mat-1-0_up.png
input/7/2486-mat-1-0_up.png
input/7/1323-mat-0-0_up.png
input/7/2952-mat-2-3.png
input/7/2096-mat-2-0.png
input/7/3191-mat-2-0.png
input/7/298-mat-1-0_up.png
input/7/521-mat-1-1.png
input/7/150-mat-2-0.png
input/7/3059-mat-0-0_up.png
input/7/238-mat-1-0_up.png
input/7/2624-mat-0-0_up.png



In [ ]:
import os, subprocess, json, re
from huggingface_hub import HfApi, hf_hub_download
from google.colab import userdata

os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
repo, KEEP, OUT = "yzwang/X2I-subject-driven", ["single","double","triple"], "/content/imgs"
keywords = ["desktop computer","computer setup","laptop","tablet","smartphone","mobile phone",
    "cell phone","headphones","earphones","earbuds","keyboard","computer mouse","game controller",
    "gaming controller","gaming console","vr headset","virtual reality headset"]
pat = re.compile("|".join(re.escape(k) for k in keywords), re.IGNORECASE)

path = hf_hub_download(repo, "grit-entity-new.jsonl", repo_type="dataset")
expected = {d: set() for d in KEEP}
for line in open(path, encoding="utf-8"):
    row = json.loads(line)
    if not pat.search(str(row.get("instruction",""))): continue
    paths = [p for p in row.get("input_images",[]) + [row.get("output_image","")] if p]
    dirs = {p.split("/")[0] for p in paths}
    if not dirs or not dirs.issubset(set(KEEP)): continue
    for p in paths:
        top, rest = p.split("/", 1)
        expected[top].add(rest)
for d in KEEP:
    os.makedirs(f"{OUT}/{d}", exist_ok=True)
    open(f"members_{d}.txt","w").write("\n".join(sorted(expected[d]))+"\n")

api = HfApi(); all_files = api.list_repo_files(repo, repo_type="dataset")
parts_of = lambda d: [f for f in sorted(all_files) if f.startswith(f"{d}/{d}.tar.gz.")]

print("Preflight (first part of each archive):")
for d in KEEP:
    url = f"https://huggingface.co/datasets/{repo}/resolve/main/{parts_of(d)[0]}"
    out = subprocess.run(f'curl -sL -H "Authorization: Bearer $HF_TOKEN" "{url}" | tar -tz 2>/dev/null',
                         shell=True, capture_output=True, text=True, env=os.environ).stdout
    members = {m for m in out.splitlines() if m and not m.endswith("/")}
    hits = expected[d] & members
    print(f"  {d}: {len(expected[d])} files needed | {len(hits)} of them found in first part "
          f"-> {'OK' if hits else 'NO MATCH — archive layout verification required'}")
    if hits: print("     e.g.", list(hits)[0])

Preflight (first part of each archive):
  single: 23780 files needed | 222 of them found in first part -> OK
     e.g. input/7/1017-mat-0-0_up.png
  double: 6651 files needed | 68 of them found in first part -> OK
     e.g. input/7/1024-mat-1-1.png
  triple: 6632 files needed | 26 of them found in first part -> OK
     e.g. input1/7/431-mat-2-1.png


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
for d in KEEP:
    done = sum(len(f) for _,_,f in os.walk(f"{OUT}/{d}"))
    if done >= len(expected[d]):
        print(f"{d}: already have {done} files, skipping"); continue
    parts = parts_of(d)
    urls = " ".join(f'"https://huggingface.co/datasets/{repo}/resolve/main/{p}"' for p in parts)
    print(f"{d}: streaming {len(parts)} parts, extracting {len(expected[d])} files...")
    cmd = (f'for u in {urls}; do curl -sL -H "Authorization: Bearer $HF_TOKEN" "$u"; done '
           f'| tar -xz -f - -C {OUT}/{d} -T members_{d}.txt 2>/dev/null')
    subprocess.run(cmd, shell=True, env=os.environ)
    got = sum(len(f) for _,_,f in os.walk(f"{OUT}/{d}"))
    print(f"  {d}: extracted {got}/{len(expected[d])} files")

single: streaming 109 parts, extracting 23780 files...
  single: extracted 23780/23780 files
double: already have 6651 files, skipping
triple: streaming 139 parts, extracting 6632 files...
  triple: extracted 6632/6632 files


In [ ]:
import os, subprocess, shutil

KEEP, OUT = ["single","double","triple"], "/content/imgs"
DRIVE_DIR = "/content/drive/MyDrive/Colab Notebooks/My Master's Thesis/Data"
os.makedirs(DRIVE_DIR, exist_ok=True)

# Reads the member lists written in the preflight
print("Extraction check:")
ok = True
for d in KEEP:
    got  = sum(len(f) for _,_,f in os.walk(f"{OUT}/{d}"))
    need = sum(1 for _ in open(f"members_{d}.txt"))
    short = got < need
    print(f"  {d}: {got}/{need} files" + ("   <-- SHORT" if short else ""))
    ok = ok and not short

# Tar everything into ONE local file (no gzip — PNGs are already compressed)
print("\nCreating single tar (local)...")
subprocess.run(f'tar -cf /content/imgs_subset.tar -C {OUT} .', shell=True)
print(f"  imgs_subset.tar = {os.path.getsize('/content/imgs_subset.tar')/1e9:.1f} GB")

# Copy the tar + split files to Drive
print("\nCopying to Drive...")
shutil.copy('/content/imgs_subset.tar', f"{DRIVE_DIR}/imgs_subset.tar")
for f in ["train.jsonl","val.jsonl","test.jsonl"]:
    if os.path.exists(f): shutil.copy(f, f"{DRIVE_DIR}/{f}")

print(f"\nSaved to {DRIVE_DIR}")
print("DONE." if ok else "DONE — but a folder was SHORT; Cell B may not have finished (safe to re-run B, it resumes).")

Extraction check:
  single: 23780/23780 files
  double: 6651/6651 files
  triple: 6632/6632 files

Creating single tar (local)...
  imgs_subset.tar = 48.2 GB

Copying to Drive...

Saved to /content/drive/MyDrive/Colab Notebooks/My Master's Thesis/Data
DONE.


In [ ]:
import os, subprocess
TAR = "/content/drive/MyDrive/Colab Notebooks/My Master's Thesis/Data/imgs_subset.tar"
print("Drive tar size (GB):", round(os.path.getsize(TAR)/1e9, 2), " (expect ~48.2)")
r = subprocess.run(f'tar -tf "{TAR}" 2>/dev/null | grep -c "\\.png$"', shell=True, capture_output=True, text=True)
print("PNG entries in Drive tar:", r.stdout.strip(), " (expect 37063)")

Drive tar size (GB): 48.18  (expect ~48.2)
PNG entries in Drive tar: 37063  (expect 37063)


In [ ]:
import os, subprocess, json

DRIVE = "/content/drive/MyDrive/Colab Notebooks/My Master's Thesis/Data"
TAR, IMGS = f"{DRIVE}/imgs_subset.tar", "/content/imgs"
os.makedirs(IMGS, exist_ok=True)

# clean restore from the (now complete) Drive tar
subprocess.run(f'cp "{TAR}" /content/imgs_subset.tar && tar -xf /content/imgs_subset.tar -C {IMGS}',
               shell=True, check=True)
print("Local image files:", sum(len(f) for _,_,f in os.walk(IMGS)), "(expect 37063)\n")

# prove every image the splits need is present
for s in ["train","val","test"]:
    if os.path.exists(f"{DRIVE}/{s}.jsonl"):
        subprocess.run(["cp", f"{DRIVE}/{s}.jsonl", f"/content/{s}.jsonl"])
total = 0
for s in ["train","val","test"]:
    rows = [json.loads(l) for l in open(f"/content/{s}.jsonl", encoding="utf-8")]
    miss = sum(1 for r in rows for p in r.get("input_images",[]) + [r.get("output_image","")]
               if p and not os.path.exists(os.path.join(IMGS, p)))
    total += miss
    print(f"{s}: {len(rows)} rows, {miss} missing images")
print("\nTOTAL missing:", total, "->", "COMPLETE — every needed image is on disk" if total == 0 else "SHORT")

Local image files: 37063 (expect 37063)

train: 39893 rows, 0 missing images
val: 98 rows, 0 missing images
test: 488 rows, 0 missing images

TOTAL missing: 0 -> COMPLETE — every needed image is on disk
